# 17 LABS Exploration

This notebook is a lightweight exploration layer on top of the refactored project modules.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from config import (
    COMMODITY_TICKERS,
    CRYPTO_TICKERS,
    NUM_PORTFOLIOS,
    RANDOM_SEED,
    RISK_FREE_RATE,
    RISK_PROFILES,
    SELECTED_PROFILE,
    START_DATE,
    TARGET_RETURN,
    TARGET_TOLERANCE,
    US_TICKERS,
    VN_TICKERS,
)
from src.asset_ranking import rank_assets_by_sharpe
from src.data_loader import get_multi_asset_data
from src.optimization import simulate_portfolios
from src.risk_analysis import analyze_return_distribution
from src.target_portfolio import select_target_return_portfolio
from src.visualization import plot_asset_ranking, plot_return_distribution, plot_simulation_frontier

In [ ]:
prices = get_multi_asset_data(
    vn_tickers=VN_TICKERS,
    us_tickers=US_TICKERS,
    crypto_tickers=CRYPTO_TICKERS,
    commodity_tickers=COMMODITY_TICKERS,
    start_date=START_DATE,
)
prices.tail()

In [ ]:
ranking = rank_assets_by_sharpe(prices, rf_rate=RISK_FREE_RATE)
plot_asset_ranking(ranking, rf_rate=RISK_FREE_RATE)
ranking.round(4)

In [ ]:
simulation = simulate_portfolios(
    prices=prices,
    risk_budget=RISK_PROFILES[SELECTED_PROFILE],
    stock_tickers=VN_TICKERS + US_TICKERS,
    crypto_tickers=CRYPTO_TICKERS,
    commodity_tickers=COMMODITY_TICKERS,
    rf_rate=RISK_FREE_RATE,
    num_portfolios=NUM_PORTFOLIOS,
    random_seed=RANDOM_SEED,
)

target_portfolio = select_target_return_portfolio(
    simulation=simulation,
    target_return=TARGET_RETURN,
    tolerance=TARGET_TOLERANCE,
)

selected_portfolio = target_portfolio or simulation.best_portfolio
distribution = analyze_return_distribution(
    expected_return=selected_portfolio.expected_return,
    volatility=selected_portfolio.volatility,
)

plot_simulation_frontier(simulation, target_portfolio=target_portfolio)
plot_return_distribution(
    distribution,
    title="Target Portfolio Return Distribution" if target_portfolio else "Max-Sharpe Return Distribution",
)

selected_portfolio.weights.sort_values(ascending=False)